# 01 — First look at the London crime data

Goal of this notebook: load the raw CSVs and understand what we're actually
working with — shape, columns, dtypes, missing values, date coverage — before
we do any real analysis. No conclusions yet, just honest observation.

In [ ]:
# sys.path.append lets us import from ../src, since notebooks/ and src/
# are sibling folders (neither is "inside" the other).
import sys
sys.path.append("../src")

import pandas as pd
from load_data import load_force_data

london = load_force_data("london")
london.shape

`.shape` gives (rows, columns) — a quick sanity check that we got roughly
what we expected (13 months of ~90-100k rows each).

In [ ]:
london.head()

In [ ]:
# .info() shows column names, dtypes, and non-null counts in one go —
# usually the first thing worth running on any new DataFrame.
london.info()

In [ ]:
# .isna() marks each cell True/False for missing; .sum() adds those up
# per column (True counts as 1). This tells us which columns actually
# have gaps, and how big they are.
london.isna().sum()

### Findings (confirmed, not assumed)
- `Crime ID` is missing for ~21% of rows, and **every single one is
  "Anti-social behaviour"** — confirmed by grouping Crime type where
  Crime ID is null. This is deliberate anonymisation by the police, not
  a loading bug. `Last outcome category` is null for the exact same rows,
  since ASB incidents don't get formal tracked outcomes.
- ~6,899 Crime IDs appear more than once. Checked an example: same ID,
  same month, but two different Crime types ("Other theft" and
  "Criminal damage and arson"). This is one real incident filed under
  multiple offence categories — not double-counted data. Decide
  deliberately later whether to keep both rows (every offence counted)
  or collapse to one row per incident, depending on the question.
- `Context` is 100% empty across all 1.2M rows — safe to drop.

In [ ]:
# .unique() lists distinct values. For Month, this confirms exactly which
# months made it into the combined DataFrame, in what looks like order
# (though unique() doesn't guarantee sorted output).
sorted(london["Month"].unique())

In [ ]:
# Are any Crime IDs duplicated? Real crimes shouldn't appear twice.
# We exclude blank IDs first since many rows share the same "missing" value,
# which would otherwise look like massive duplication.
has_id = london["Crime ID"].dropna()
has_id.duplicated().sum()

In [ ]:
# value_counts() tallies how often each distinct value appears —
# here, which crime types are most common in London.
london["Crime type"].value_counts()